# Week 15, Day 4: Recursion on strings and lists

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w015d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def length(items):
    if not items:
        return 0
    return 1 + length(items[1:])

print(length([7, 8, 9]), length("hello"), length([]))

In [ ]:
def total(items):
    if not items:
        return 0
    return items[0] + total(items[1:])

def contains(items, target):
    if not items:
        return False
    return items[0] == target or contains(items[1:], target)

def count_vowels(text):
    if not text:
        return 0
    return (text[0].lower() in "aeiou") + count_vowels(text[1:])

print(total([4, 5, 6]), contains([4, 5, 6], 5), contains([], 5), count_vowels("Recursion"))

In [ ]:
def is_palindrome(text):
    if len(text) < 2:
        return True
    return text[0] == text[-1] and is_palindrome(text[1:-1])

for word in ["level", "racecar", "python", "a", ""]:
    print(word, is_palindrome(word))

> **Careful.** Each `seq[1:]` makes a new copy of the rest, so these functions do extra copying, and they go one level deeper per item. They're perfect for learning and for short sequences, but for a list of 10,000 items, use a loop. Day 5 compares the two.

In [ ]:
def doubled(items):
    if not items:
        return []
    return [items[0] * 2] + doubled(items[1:])

def only_evens(items):
    if not items:
        return []
    rest = only_evens(items[1:])
    return [items[0]] + rest if items[0] % 2 == 0 else rest

def remove_spaces(text):
    if not text:
        return ""
    return ("" if text[0] == " " else text[0]) + remove_spaces(text[1:])

print(doubled([1, 2, 3]), only_evens([1, 2, 3, 4]), remove_spaces("a b  c"))

In [ ]:
def nested_sum(items):
    total = 0
    for item in items:
        if isinstance(item, list):
            total += nested_sum(item)
        else:
            total += item
    return total

print(nested_sum([1, [2, 3], [4, [5, [6]]], 7]))

In [ ]:
def flatten(items):
    flat = []
    for item in items:
        if isinstance(item, list):
            flat.extend(flatten(item))
        else:
            flat.append(item)
    return flat

print(flatten([1, [2, 3], [4, [5, [6]]], 7]))

*Interactive figure on the website: One call per list, at every depth. Each call returns its flat list to the call that was waiting for it..*

> **Key idea.** For nested data, the recursion follows the data: when an item is itself a container, call the same function on it. One function handles every depth.

In [ ]:
folder = {
    "notes.txt": 120,
    "photos": {
        "beach.jpg": 3400,
        "family": {"gran.jpg": 2900, "cake.jpg": 1800},
    },
    "code": {"atlas.py": 45, "old": {}},
}

def folder_size(folder):
    size = 0
    for name, item in folder.items():
        if isinstance(item, dict):
            size += folder_size(item)
        else:
            size += item
    return size

def list_files(folder, path=""):
    paths = []
    for name, item in folder.items():
        full = f"{path}/{name}"
        if isinstance(item, dict):
            paths += list_files(item, full)
        else:
            paths.append(full)
    return paths

print(folder_size(folder))
for p in list_files(folder):
    print(p)

In [ ]:
data = {
    "name": "Atlas",
    "authors": [{"name": "Ada"}, {"name": "Bo", "pet": {"name": "Rex"}}],
    "meta": {"version": 2},
}

def find_all(data, key):
    found = []
    if isinstance(data, dict):
        for k, v in data.items():
            if k == key:
                found.append(v)
            found += find_all(v, key)
    elif isinstance(data, list):
        for item in data:
            found += find_all(item, key)
    return found

print(find_all(data, "name"))

### Exercise: Reverse a list

Write `reversed_list(items)` that returns a new list with the items in reverse order, recursively, using the first-and-rest shape (no loops, no `reverse`, no `[::-1]`).

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert reversed_list([1, 2, 3]) == [3, 2, 1] and reversed_list([]) == [] and reversed_list([\"a\"]) == [\"a\"]\n    assert \"for \" not in source and \"while \" not in source and \"[::-1]\" not in source and \".reverse(\" not in source")

### Exercise: Interleave

Write `interleave(a, b)` that alternates the items of two lists, starting with `a`, and adds whatever is left of the longer one at the end: `interleave([1, 2, 3], ["x", "y"])` is `[1, "x", 2, "y", 3]`. Recursively, with no loops.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert interleave([1, 2, 3], [\"x\", \"y\"]) == [1, \"x\", 2, \"y\", 3]\n    assert interleave([], [1, 2]) == [1, 2] and interleave([1], []) == [1] and interleave([], []) == []\n    assert \"for \" not in source and \"while \" not in source")

### Exercise: How deep?

Write `depth(items)` that returns how deeply a list is nested: a list with no lists inside it has depth 1; `[1, [2, [3]]]` has depth 3; the empty list has depth 1. Anything that isn't a list has depth 0.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert depth([1, 2]) == 1 and depth([]) == 1 and depth([1, [2, [3]]]) == 3\n    assert depth([[], [[]], [[[]]]]) == 4 and depth(5) == 0")

### Exercise: Count everywhere

Write `deep_count(items, target)` that counts how many times `target` appears anywhere in a nested list.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert deep_count([1, [1, 2, [1]], 3], 1) == 3\n    assert deep_count([[[\"x\"]], \"x\", [\"y\"]], \"x\") == 2\n    assert deep_count([], 1) == 0")

### Exercise: Same shape, new values

Write `deep_map(f, items)` that returns a new nested list of the same shape, with `f` applied to every non-list item: `deep_map(str, [1, [2, [3]]])` is `["1", ["2", ["3"]]]`. Don't change the original.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    v = [1, [2, [3, 4]], []]\n    assert deep_map(lambda x: x * 10, v) == [10, [20, [30, 40]], []]\n    assert v == [1, [2, [3, 4]], []]\n    assert deep_map(str, [1, [2, [3]]]) == [\"1\", [\"2\", [\"3\"]]]")

### Exercise: The biggest file

Using a folder as nested dictionaries (like the example), write `biggest_file(folder)` that returns a tuple `(path, size)` for the largest file anywhere inside, with paths written like `/photos/family/gran.jpg`. Return `None` if there are no files at all.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("FOLDER = {\n    \"notes.txt\": 120,\n    \"photos\": {\"beach.jpg\": 3400, \"family\": {\"gran.jpg\": 2900, \"cake.jpg\": 1800}},\n    \"code\": {\"atlas.py\": 45, \"old\": {}, \"big.bin\": 5000},\n}\ndef test_values():\n    \"several\"\n    assert biggest_file(FOLDER) == (\"/code/big.bin\", 5000)\n    assert biggest_file({\"a\": {\"b\": {}}}) is None\n    assert biggest_file({\"x\": {\"y\": {\"z.txt\": 1}}}) == (\"/x/y/z.txt\", 1)")

### Bug hunt: Only one level

`flatten` should flatten a list nested to any depth, but it only goes one level down: `[1, [2, [3]]]` gives `[1, 2, [3]]`. Fix it.

In [ ]:
def flatten(items):
    flat = []
    for item in items:
        if isinstance(item, list):
            flat.extend(item)
        else:
            flat.append(item)
    return flat

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"every depth\"\n    assert flatten([1, [2, [3]]]) == [1, 2, 3]\n    assert flatten([[[[]]]]) == [] and flatten([]) == []")

### Bug hunt: Lost on the way up

`only_positive(items)` should return the positive numbers, in order, recursively. It returns only the first one. Fix the recursive case.

In [ ]:
def only_positive(items):
    if not items:
        return []
    if items[0] > 0:
        return [items[0]]
    return only_positive(items[1:])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert only_positive([3, -1, 4, 0, 5]) == [3, 4, 5] and only_positive([-2]) == [] and only_positive([]) == []")

### Puzzle: Find a value in nested data

Arrange the lines so `contains(data, target)` checks whether `target` appears anywhere in nested lists. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # flatten([[1, [2]], [], [[3, [4, []]]]])
b = ___   # depth([[1], [[2]], 3])   (today's depth)
c = ___   # is_palindrome("abca")
d = ___   # nested_sum([[[]], [1, [1, [1]]]])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [1, 2, 3, 4]\n    assert b == 3\n    assert c is False\n    assert d == 3")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: pretty-printing nested data

Write `pretty(data)` that returns a string showing nested dictionaries and lists as an indented outline, two spaces per level:

- a dictionary shows each key followed by a colon; if the value is a dictionary or a list, it goes on the following lines, indented; otherwise it follows on the same line after a space;
- a list shows each item after `- `; if an item is a dictionary or list, it goes on the following lines, indented under the `-`;
- other values are shown with `repr`.

```
pretty({"name": "Atlas", "tags": ["py", "free"], "meta": {"v": 2}})  gives:

name: 'Atlas'
tags:
  - 'py'
  - 'free'
meta:
  v: 2
```

Empty dictionaries and lists are shown as `{}` and `[]` on the same line.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_example():\n    \"the example\"\n    assert pretty({\"name\": \"Atlas\", \"tags\": [\"py\", \"free\"], \"meta\": {\"v\": 2}}) == \"name: 'Atlas'\\ntags:\\n  - 'py'\\n  - 'free'\\nmeta:\\n  v: 2\"\ndef test_deeper():\n    \"deeper\"\n    d = {\"a\": [1, {\"b\": [], \"c\": {}}, [2, 3]]}\n    assert pretty(d) == \"a:\\n  - 1\\n  -\\n    b: []\\n    c: {}\\n  -\\n    - 2\\n    - 3\"\ndef test_top_level():\n    \"a list at the top\"\n    assert pretty([1, \"x\"]) == \"- 1\\n- 'x'\"\n    assert pretty({}) == \"{}\" and pretty(5) == \"5\"")

## Warm-up

A list is either **empty**, or **a first item followed by a smaller list**. Every recursive function on lists follows that shape, and `length` is the simplest of them.

## First and rest

Most recursive functions on a sequence look at `seq[0]`, the **first** item, and recurse on `seq[1:]`, **the rest**:

```
def f(seq):
    if not seq:                       # the empty sequence
        return the answer for nothing
    first, rest = seq[0], seq[1:]
    return combine(first, f(rest))    # leap of faith: f(rest) is right
```

Here are several functions you've written with loops, now in this shape:

`(text[0].lower() in "aeiou")` is `True` or `False`, which count as 1 and 0 when added.

Palindromes have a different but equally natural shape: **first and last, and the middle**:

## Building results

When the result is a new sequence, combine the first item's contribution with the result for the rest:

Those are the transform, filter and map patterns of Week 14, written recursively. The leap of faith makes each one obvious: "if I had the answer for the rest, I'd put this item's part in front of it."

## Nested lists

Now data that is recursive itself. A **nested list** holds numbers, and other nested lists, to any depth:

```
[1, [2, 3], [4, [5, [6]]], 7]
```

With loops alone this is awkward, because you don't know how many levels of loops you need. With recursion it's easy, because each item is either a **number** (handle it directly) or a **list** (which is the same kind of problem, smaller):

Notice this mixes a loop (across one level) with recursion (down into each sub-list). That's normal and usually clearest for nested data: loop over the items at this level, recurse into the ones that are themselves nested.

**Flattening** a nested list into a flat one follows the same shape:

Watch it go down into the sub-lists and come back:

## Trees of data

The same idea works for anything that contains smaller things of the same kind. A **folder** contains files and folders. As nested dictionaries, with a file's size as a number:

`list_files` passes the **path so far** down to each recursive call, so every file knows where it is. Passing information down as an argument, and results back up as a return value, is how recursive functions communicate.

JSON from Week 10 has exactly this shape: dictionaries and lists inside each other, to any depth. Here's a function that finds every value for a given key, wherever it is:

## Practice

## Quiz

## Daily challenge

## Before you go

Look for recursive data in your own life: folders on your computer, a family tree, a contents page with chapters and sections, comments with replies to replies. For one of them, sketch the nested dictionaries or lists and a recursive function to count everything in it. Tomorrow: recursion versus loops, and how to choose.